In [7]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np

import torch
from torch import nn
from torch.utils.data import TensorDataset

import importlib
import train_helper

from MLP import MLP


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/hoangmacmini/Library/Mobile Documents/com~apple~CloudDocs/Documents/Notes/ML Coding/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/hoangmacmini/Library/Mobile Documents/com~apple~CloudDocs/Documents/Notes/ML Coding/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  

In [2]:
importlib.reload(train_helper)

<module 'train_helper' from '/Users/hoangmacmini/Library/Mobile Documents/com~apple~CloudDocs/Documents/Notes/ML Coding/train_helper.py'>

In [3]:
# ===== Cell 2: Small helper metrics (optional) =====
def mean_squared_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    diff = y_true.reshape(-1) - y_pred.reshape(-1)
    return float(np.mean(diff * diff))

def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    return float(np.mean(y_true == y_pred))

In [4]:
# ===== Cell 3: MLP regression example (tabular) =====
np.random.seed(0)

sample_count = 1000
input_layer_size = 10

X_all = np.random.randn(sample_count, input_layer_size).astype(np.float32)
y_all = (2.0 * X_all[:, 0] - 1.0 * X_all[:, 1] + 0.1 * np.random.randn(sample_count)).astype(np.float32)

train_count = 800
X_train_np, X_test_np = X_all[:train_count], X_all[train_count:]
y_train_np, y_test_np = y_all[:train_count], y_all[train_count:]


In [5]:
X_train = train_helper.build_tensor(X_train_np)
X_test  = train_helper.build_tensor(X_test_np)
y_train = train_helper.build_tensor(y_train_np).view(-1, 1)

training_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test)

In [10]:
hidden_layer_sizes = [64, 64]
model = MLP(input_layer_size, hidden_layer_sizes, 1)

train_helper.fit(
    X_and_y = training_data,
    model = model,
    loss_func = nn.MSELoss(),
    optimizing_func = torch.optim.Adam,
    lr = 1e-3,
    max_epoch = 10,
    batch_size = 128
)

MLP(
  (hidden_layers): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)

In [11]:
y_pred_tensor = train_helper.predict(test_data, model, batch_size=256)   # torch.Tensor (N,1)
y_test_tensor = train_helper.build_tensor(y_test_np).view(-1, 1)

mse_tensor = torch.mean((y_pred_tensor - y_test_tensor) ** 2)
print("MSE =", float(mse_tensor.item()))


MSE = 0.2150387167930603
